# NB-04 | Benchmark: Baseline 4 — LLM Prompt Expansion + Character LoRA

Оценка **BL2 + LLM auto-expansion** (Ollama).  
Ключевой вопрос: улучшает ли LLM-расширение промптов CLIP Score?  
При этом CLIP Score считается по **расширенному** промпту, а не по оригинальному.

In [1]:
import os, json, struct
from pathlib import Path
import torch, numpy as np, mlflow
from PIL import Image
from torchmetrics.image.kid import KernelInceptionDistance
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity
import open_clip
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

e:\Projects\Auto-VideoGame-Assets-Pipeline\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


In [2]:
BASELINE_ID   = 'baseline_4'
GENERATED_DIR = Path('generated/baseline_4')
REFERENCE_DIR = Path('reference/characters')

# Расширенные промпты, сохранённые NB-00
EXPANDED_PROMPTS_JSON = Path('generated/baseline_4/expanded_prompts.json')

MLFLOW_URI      = 'http://127.0.0.1:5000'
EXPERIMENT_NAME = 'anima_baseline_benchmarks'

MODEL_PARAMS = {
    'model.name':          'anima_base',
    'model.type':          'character_lora_llm',
    'model.char_lora':     'sltn_character_v2.safetensors',
    'model.trigger':       '@sltn',
    'model.lora_stack':    'turbo+character',
}

INFER_PARAMS = {
    'infer.baseline_id':    BASELINE_ID,
    'infer.sampler':        'euler_ancestral',
    'infer.scheduler':      'normal',
    'infer.steps':          12,
    'infer.cfg':            2.0,
    'infer.width':          512,
    'infer.height':         512,
    'infer.turbo_strength': 1.0,
    'infer.style_strength': 0.85,
    'infer.llm_expansion':  True,
    'infer.llm_model':      'mistral',
    'infer.llm_url':        'http://127.0.0.1:11434',
    'infer.seed_policy':    'idx*7919',
    'infer.n_images':       100,
}

print('Config loaded ✓')

Config loaded ✓


In [3]:
# Загружаем расширенные промпты из NB-00
if EXPANDED_PROMPTS_JSON.exists():
    with open(EXPANDED_PROMPTS_JSON) as f:
        expanded_data = json.load(f)
    # CLIP считаем по expanded_prompt, не по raw
    PROMPTS_FOR_CLIP = [d['expanded_prompt'] for d in expanded_data]
    print(f'Expanded prompts loaded: {len(PROMPTS_FOR_CLIP)}')
    print(f'Example: {PROMPTS_FOR_CLIP[0][:80]}...')
else:
    print('WARNING: expanded_prompts.json not found — falling back to raw prompts')
    BASE_PROMPTS = [
        '1girl, solo, long hair, blue eyes, smile, outdoors, sky',
        '1girl, solo, short hair, red eyes, serious, city background',
        '1girl, solo, twin tails, green eyes, school uniform, outdoors',
        '1girl, solo, white hair, purple eyes, simple background',
        '1girl, solo, ponytail, brown eyes, jacket, forest',
        '1boy, solo, short hair, blue eyes, armor, standing',
        '1girl, solo, black hair, dress, indoors',
        '1girl, solo, cat ears, pink hair, happy, outdoors',
        '1boy, solo, messy hair, red scarf, city',
        '1girl, solo, silver hair, golden eyes, cape, sky background',
    ]
    PROMPTS_FOR_CLIP = [BASE_PROMPTS[i % 10] for i in range(100)]

Expanded prompts loaded: 100
Example: @sltn, 1girl, solo, long hair, blue eyes, smile, outdoors, sky, highly detailed ...


In [4]:
def load_t(folder, size=(299,299)):
    return torch.stack([torch.tensor(np.array(Image.open(p).convert('RGB').resize(size))).permute(2,0,1) for p in sorted(folder.glob('*.png'))])

def load_pil(folder, limit=None):
    files = sorted(folder.glob('*.png'))
    if limit: files = files[:limit]
    return [Image.open(p).convert('RGB') for p in files]

def make_grid(imgs, n=8, thumb=128):
    imgs=[i.resize((thumb,thumb)) for i in imgs[:n]]
    c=Image.new('RGB',(thumb*len(imgs),thumb),(20,20,20))
    for i,im in enumerate(imgs): c.paste(im,(thumb*i,0))
    return c

def compute_kid(gen, ref, subset=50):
    k=KernelInceptionDistance(subset_size=subset,normalize=True).to(DEVICE)
    k.update(load_t(ref).to(DEVICE),real=True)
    k.update(load_t(gen).to(DEVICE),real=False)
    m,s=k.compute(); return float(m.cpu()),float(s.cpu())

def compute_clip(gen, prompts):
    model,_,prep=open_clip.create_model_and_transforms('ViT-B-32',pretrained='openai')
    model=model.to(DEVICE).eval()
    tok=open_clip.get_tokenizer('ViT-B-32')
    sc=[]
    for i,p in enumerate(tqdm(sorted(gen.glob('*.png')),'CLIP')):
        if i>=len(prompts): break
        img=prep(Image.open(p).convert('RGB')).unsqueeze(0).to(DEVICE)
        txt=tok([prompts[i]]).to(DEVICE)
        with torch.no_grad():
            a=model.encode_image(img);a/=a.norm(dim=-1,keepdim=True)
            b=model.encode_text(txt);b/=b.norm(dim=-1,keepdim=True)
            sc.append(float((a*b).sum().cpu()))
    return float(np.mean(sc))

def compute_lpips(gen, n_pairs=200):
    lp=LearnedPerceptualImagePatchSimilarity(net_type='vgg',normalize=True).to(DEVICE)
    files=sorted(gen.glob('*.png'))
    idx=np.random.default_rng(42).integers(0,len(files),size=(n_pairs,2))
    sc=[]
    def tt(p): return torch.tensor(np.array(Image.open(p).convert('RGB').resize((256,256))).astype(np.float32)/255.).permute(2,0,1).unsqueeze(0).to(DEVICE)
    for a,b in tqdm(idx,'LPIPS'):
        if a==b: continue
        with torch.no_grad(): sc.append(float(lp(tt(files[a]),tt(files[b])).cpu()))
    return float(np.mean(sc))

print('Helpers ready ✓')

Helpers ready ✓


In [5]:
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

run_name = f'eval_{BASELINE_ID}_charLoRA_llmExpand'

with mlflow.start_run(run_name=run_name) as run:
    print(f'Run ID: {run.info.run_id}')

    mlflow.set_tags({
        'evaluation_type': 'benchmark',
        'training_logged_retroactively': 'true',
        'family':          'character',
        'baseline_id':     BASELINE_ID,
        'llm_expansion':   'true',
        'environment':     'local_rtx4060ti_16gb',
    })

    mlflow.log_params(MODEL_PARAMS)
    mlflow.log_params(INFER_PARAMS)
    mlflow.log_param('data.clip_evaluated_on', 'expanded_prompts')

    # Логируем expanded_prompts.json как artifact
    if EXPANDED_PROMPTS_JSON.exists():
        mlflow.log_artifact(str(EXPANDED_PROMPTS_JSON), artifact_path='prompts')

    kid_mean, kid_std = compute_kid(GENERATED_DIR, REFERENCE_DIR)
    clip_score        = compute_clip(GENERATED_DIR, PROMPTS_FOR_CLIP)
    lpips_div         = compute_lpips(GENERATED_DIR)

    mlflow.log_metrics({
        'eval.kid_mean':        kid_mean,
        'eval.kid_std':         kid_std,
        'eval.clip_score':      clip_score,
        'eval.lpips_diversity': lpips_div,
        'eval.n_generated':     float(len(list(GENERATED_DIR.glob('*.png')))),
    })

    grid = make_grid(load_pil(GENERATED_DIR, limit=8))
    Path('tmp').mkdir(parents=True, exist_ok=True)
    gp = f'tmp/{BASELINE_ID}_grid.png'; grid.save(gp)
    mlflow.log_artifact(gp, artifact_path='samples')

    print(f'\n=== BL4 Results ===')
    print(f'KID:   {kid_mean:.6f} ± {kid_std:.6f}')
    print(f'CLIP:  {clip_score:.4f}  (measured on expanded prompts)')
    print(f'LPIPS: {lpips_div:.4f}')
    print('✓ Logged to MLflow')

Run ID: e62d0d92a4544009a9eaafa3649c595a


e:\Projects\Auto-VideoGame-Assets-Pipeline\venv\lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: Metric `Kernel Inception Distance` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
e:\Projects\Auto-VideoGame-Assets-Pipeline\venv\lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(
LPIPS: 100%|██████████| 200/200 [01:04<00:00,  3.12it/s]


=== BL4 Results ===
KID:   0.056798 ± 0.005602
CLIP:  0.2922  (measured on expanded prompts)
LPIPS: 0.6514
✓ Logged to MLflow
🏃 View run eval_baseline_4_charLoRA_llmExpand at: http://127.0.0.1:5000/#/experiments/1/runs/e62d0d92a4544009a9eaafa3649c595a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
